# ML Experiment 2 (A2) - Regression Analysis
### Option B: Diabetes (UCI) and Pima Indians Diabetes datasets

**Aim:** Perform univariate analysis, bivariate analysis (linear and logistic regression), and multiple regression on the Pima Indians and UCI diabetes datasets, then compare the results.

**Step 1:** Import the required libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix
from sklearn.datasets import load_diabetes

sns.set(style="whitegrid")
print("Libraries ready")

## PART A: Pima Indians Diabetes Dataset

**Step 2:** Import the Pima Indians Diabetes dataset and add column names.

In [ ]:
cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin',
        'BMI','DiabetesPedigreeFunction','Age','Outcome']
url = "pima-indians-diabetes.csv"
pima = pd.read_csv(url, header=None, names=cols)
pima.head()

**Step 3 (a) - Univariate Analysis:** Frequency, Mean, Median, Mode, Variance, Standard Deviation, Skewness, and Kurtosis for each column.

In [ ]:
univariate = pd.DataFrame({
    "Mean": pima.mean(),
    "Median": pima.median(),
    "Mode": pima.mode().iloc[0],
    "Variance": pima.var(),
    "Std_Dev": pima.std(),
    "Skewness": pima.skew(),
    "Kurtosis": pima.kurt()
})
univariate.round(3)

In [ ]:
print("Frequency of Outcome (0 = No diabetes, 1 = Diabetes):")
print(pima["Outcome"].value_counts())

pima["Glucose"].hist(bins=20, color="skyblue", edgecolor="black")
plt.title("Frequency Distribution of Glucose")
plt.xlabel("Glucose"); plt.ylabel("Frequency")
plt.show()

**Step 4 (b) - Bivariate Analysis: Logistic Regression.** Predict whether a patient has diabetes (Outcome) from all other features.

In [ ]:
X = pima.drop("Outcome", axis=1)
y = pima["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

print("Logistic Regression Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

**Step 5 (b) - Bivariate Analysis: Linear Regression.** Fit a simple linear regression between two variables (Glucose vs BMI).

In [ ]:
lin_model = LinearRegression()
lin_model.fit(pima[["Glucose"]], pima["BMI"])

print("R2 score (Glucose -> BMI):", round(lin_model.score(pima[["Glucose"]], pima["BMI"]), 3))

plt.figure(figsize=(7, 5))
plt.scatter(pima["Glucose"], pima["BMI"], alpha=0.4)
plt.plot(pima["Glucose"], lin_model.predict(pima[["Glucose"]]), color="red")
plt.title("Linear Regression: Glucose vs BMI")
plt.xlabel("Glucose"); plt.ylabel("BMI")
plt.show()

**Step 6 (c) - Multiple Regression.** Predict BMI using several features together, and evaluate with R2 and RMSE.

In [ ]:
features = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','Age']
Xm = pima[features]
ym = pima["BMI"]

Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.2, random_state=42)

multi_model = LinearRegression()
multi_model.fit(Xm_train, ym_train)
ym_pred = multi_model.predict(Xm_test)

pima_r2 = r2_score(ym_test, ym_pred)
pima_rmse = np.sqrt(mean_squared_error(ym_test, ym_pred))
print("Pima Multiple Regression -> R2:", round(pima_r2, 3), "| RMSE:", round(pima_rmse, 3))

## PART B: UCI Diabetes Dataset (built into scikit-learn)

**Step 7:** Load the UCI diabetes dataset. Here the target is a continuous measure of disease progression.

In [ ]:
uci = load_diabetes(as_frame=True)
uci_df = uci.frame
print("Shape:", uci_df.shape)
uci_df.head()

**Step 8 - Univariate Analysis** on the UCI dataset (quick summary).

In [ ]:
uci_stats = pd.DataFrame({
    "Mean": uci.data.mean(),
    "Std_Dev": uci.data.std(),
    "Skewness": uci.data.skew(),
    "Kurtosis": uci.data.kurt()
})
uci_stats.round(3)

**Step 9 - Multiple Regression** on the UCI dataset: predict disease progression from all features, evaluate with R2 and RMSE.

In [ ]:
Xu_train, Xu_test, yu_train, yu_test = train_test_split(uci.data, uci.target, test_size=0.2, random_state=42)

uci_model = LinearRegression()
uci_model.fit(Xu_train, yu_train)
yu_pred = uci_model.predict(Xu_test)

uci_r2 = r2_score(yu_test, yu_pred)
uci_rmse = np.sqrt(mean_squared_error(yu_test, yu_pred))
print("UCI Multiple Regression -> R2:", round(uci_r2, 3), "| RMSE:", round(uci_rmse, 3))

**Step 10 (d) - Compare the results** of the regression analysis for the two datasets.

In [ ]:
comparison = pd.DataFrame({
    "Dataset": ["Pima (predict BMI)", "UCI (predict progression)"],
    "R2_Score": [round(pima_r2, 3), round(uci_r2, 3)],
    "RMSE": [round(pima_rmse, 3), round(uci_rmse, 3)]
})
print(comparison.to_string(index=False))
print("\nHigher R2 = better fit. The dataset with the higher R2 is explained better by its features.")